In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil

base_dir = '/content/drive/MyDrive/Obesity_Quantum_Project'

# Ana kanit klasoru
evidence_dir = os.path.join(base_dir, 'quantum_circuit_evidence')
evidence_fig_dir = os.path.join(evidence_dir, 'figures')
evidence_code_dir = os.path.join(evidence_dir, 'code')
evidence_note_dir = os.path.join(evidence_dir, 'notes')

# Mevcut deney klasorleri
q12_fig_dir = os.path.join(base_dir, 'quantum_clean_top10_angle', 'figures')
q12_note_dir = os.path.join(base_dir, 'quantum_clean_top10_angle', 'notes')

q3_fig_dir = os.path.join(base_dir, 'quantum_clean_top10_hybrid_encoding', 'figures')
q3_note_dir = os.path.join(base_dir, 'quantum_clean_top10_hybrid_encoding', 'notes')

for d in [
    evidence_dir, evidence_fig_dir, evidence_code_dir, evidence_note_dir,
    q12_fig_dir, q12_note_dir, q3_fig_dir, q3_note_dir
]:
    os.makedirs(d, exist_ok=True)

print("Hazir klasorler:")
print("evidence_dir     :", evidence_dir)
print("evidence_fig_dir :", evidence_fig_dir)
print("evidence_code_dir:", evidence_code_dir)
print("evidence_note_dir:", evidence_note_dir)
print("q12_fig_dir      :", q12_fig_dir)
print("q3_fig_dir       :", q3_fig_dir)

Hazir klasorler:
evidence_dir     : /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence
evidence_fig_dir : /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures
evidence_code_dir: /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/code
evidence_note_dir: /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/notes
q12_fig_dir      : /content/drive/MyDrive/Obesity_Quantum_Project/quantum_clean_top10_angle/figures
q3_fig_dir       : /content/drive/MyDrive/Obesity_Quantum_Project/quantum_clean_top10_hybrid_encoding/figures


In [3]:
!pip install pennylane -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.4 MB/s eta 0:00:00


In [4]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from textwrap import dedent
import json
import os
import shutil

print("PennyLane version:", qml.__version__)

PennyLane version: 0.44.1


In [5]:
# ---------------------------
# SABITLER
# ---------------------------
Q1_QUBITS = 10
Q1_LAYERS = 2

Q2_QUBITS = 10
Q2_LAYERS = 4

Q3_ANGLE_QUBITS = 10
Q3_ANGLE_LAYERS = 2

Q3_AMP_QUBITS = 4
Q3_AMP_LAYERS = 2

# ---------------------------
# DEVICE'LAR
# ---------------------------
dev_q1 = qml.device("default.qubit", wires=Q1_QUBITS)
dev_q2 = qml.device("default.qubit", wires=Q2_QUBITS)
dev_q3_angle = qml.device("default.qubit", wires=Q3_ANGLE_QUBITS)
dev_q3_amp = qml.device("default.qubit", wires=Q3_AMP_QUBITS)

# ---------------------------
# Q1 CIRCUIT
# ---------------------------
@qml.qnode(dev_q1)
def q1_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q1_QUBITS), rotation='Y')
    for layer in range(Q1_LAYERS):
        for i in range(Q1_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q1_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q1_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q1_QUBITS)]

# ---------------------------
# Q2 CIRCUIT
# ---------------------------
@qml.qnode(dev_q2)
def q2_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q2_QUBITS), rotation='Y')
    for layer in range(Q2_LAYERS):
        for i in range(Q2_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q2_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q2_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q2_QUBITS)]

# ---------------------------
# Q3 ANGLE BRANCH
# ---------------------------
@qml.qnode(dev_q3_angle)
def q3_angle_branch(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q3_ANGLE_QUBITS), rotation='Y')
    for layer in range(Q3_ANGLE_LAYERS):
        for i in range(Q3_ANGLE_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q3_ANGLE_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q3_ANGLE_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q3_ANGLE_QUBITS)]

# ---------------------------
# Q3 AMPLITUDE BRANCH
# ---------------------------
@qml.qnode(dev_q3_amp)
def q3_amplitude_branch(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(Q3_AMP_QUBITS), normalize=True)
    for layer in range(Q3_AMP_LAYERS):
        for i in range(Q3_AMP_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q3_AMP_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q3_AMP_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q3_AMP_QUBITS)]

print("Q1, Q2, Q3 devre tanimlari hazir.")

Q1, Q2, Q3 devre tanimlari hazir.


In [6]:
q1_code = dedent("""
import pennylane as qml

Q1_QUBITS = 10
Q1_LAYERS = 2
dev_q1 = qml.device("default.qubit", wires=Q1_QUBITS)

@qml.qnode(dev_q1)
def q1_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q1_QUBITS), rotation='Y')
    for layer in range(Q1_LAYERS):
        for i in range(Q1_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q1_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q1_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q1_QUBITS)]
""").strip()

q2_code = dedent("""
import pennylane as qml

Q2_QUBITS = 10
Q2_LAYERS = 4
dev_q2 = qml.device("default.qubit", wires=Q2_QUBITS)

@qml.qnode(dev_q2)
def q2_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q2_QUBITS), rotation='Y')
    for layer in range(Q2_LAYERS):
        for i in range(Q2_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q2_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q2_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q2_QUBITS)]
""").strip()

q3_angle_code = dedent("""
import pennylane as qml

Q3_ANGLE_QUBITS = 10
Q3_ANGLE_LAYERS = 2
dev_q3_angle = qml.device("default.qubit", wires=Q3_ANGLE_QUBITS)

@qml.qnode(dev_q3_angle)
def q3_angle_branch(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(Q3_ANGLE_QUBITS), rotation='Y')
    for layer in range(Q3_ANGLE_LAYERS):
        for i in range(Q3_ANGLE_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q3_ANGLE_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q3_ANGLE_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q3_ANGLE_QUBITS)]
""").strip()

q3_amp_code = dedent("""
import pennylane as qml

Q3_AMP_QUBITS = 4
Q3_AMP_LAYERS = 2
dev_q3_amp = qml.device("default.qubit", wires=Q3_AMP_QUBITS)

@qml.qnode(dev_q3_amp)
def q3_amplitude_branch(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(Q3_AMP_QUBITS), normalize=True)
    for layer in range(Q3_AMP_LAYERS):
        for i in range(Q3_AMP_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(Q3_AMP_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % Q3_AMP_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(Q3_AMP_QUBITS)]
""").strip()

files_to_write = {
    "q1_circuit_definition.py": q1_code,
    "q2_circuit_definition.py": q2_code,
    "q3_angle_branch_definition.py": q3_angle_code,
    "q3_amplitude_branch_definition.py": q3_amp_code
}

for fname, content in files_to_write.items():
    with open(os.path.join(evidence_code_dir, fname), "w", encoding="utf-8") as f:
        f.write(content)

print("Kod kanitlari kaydedildi:")
for fname in files_to_write:
    print("-", fname)

Kod kanitlari kaydedildi:
- q1_circuit_definition.py
- q2_circuit_definition.py
- q3_angle_branch_definition.py
- q3_amplitude_branch_definition.py


In [7]:
# Ornek inputlar
sample_input_q1 = np.linspace(-0.8, 0.8, 10)
sample_weights_q1 = np.zeros((2, 10, 2))

sample_input_q2 = np.linspace(-0.8, 0.8, 10)
sample_weights_q2 = np.zeros((4, 10, 2))

sample_input_q3_angle = np.linspace(-0.8, 0.8, 10)
sample_weights_q3_angle = np.zeros((2, 10, 2))

sample_input_q3_amp = np.zeros(16)
sample_input_q3_amp[0] = 1.0
sample_weights_q3_amp = np.zeros((2, 4, 2))

# ASCII cizimler
q1_ascii = qml.draw(q1_circuit, decimals=2)(sample_input_q1, sample_weights_q1)
q2_ascii = qml.draw(q2_circuit, decimals=2)(sample_input_q2, sample_weights_q2)
q3_angle_ascii = qml.draw(q3_angle_branch, decimals=2)(sample_input_q3_angle, sample_weights_q3_angle)
q3_amp_ascii = qml.draw(q3_amplitude_branch, decimals=2)(sample_input_q3_amp, sample_weights_q3_amp)

ascii_files = {
    "q1_circuit_ascii.txt": q1_ascii,
    "q2_circuit_ascii.txt": q2_ascii,
    "q3_angle_branch_ascii.txt": q3_angle_ascii,
    "q3_amplitude_branch_ascii.txt": q3_amp_ascii,
}

for fname, content in ascii_files.items():
    with open(os.path.join(evidence_note_dir, fname), "w", encoding="utf-8") as f:
        f.write(content)

# İlgili deney klasorlerine de kopyala
shutil.copy(os.path.join(evidence_note_dir, "q1_circuit_ascii.txt"),
            os.path.join(q12_note_dir, "q1_circuit_ascii.txt"))

shutil.copy(os.path.join(evidence_note_dir, "q2_circuit_ascii.txt"),
            os.path.join(q12_note_dir, "q2_circuit_ascii.txt"))

shutil.copy(os.path.join(evidence_note_dir, "q3_angle_branch_ascii.txt"),
            os.path.join(q3_note_dir, "q3_angle_branch_ascii.txt"))

shutil.copy(os.path.join(evidence_note_dir, "q3_amplitude_branch_ascii.txt"),
            os.path.join(q3_note_dir, "q3_amplitude_branch_ascii.txt"))

print("ASCII devre kanitlari kaydedildi.")
print("\nQ1 ASCII onizleme:\n")
print(q1_ascii[:2000])

ASCII devre kanitlari kaydedildi.

Q1 ASCII onizleme:

0: ─╭AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╭●─────────────────────────╭X──RY(0.00)──RZ(0.00)─╭● ···
1: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╰X─╭●──────────────────────│───RY(0.00)──RZ(0.00)─╰X ···
2: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)────╰X─╭●───────────────────│───RY(0.00)──RZ(0.00)─── ···
3: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)───────╰X─╭●────────────────│───RY(0.00)──RZ(0.00)─── ···
4: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)──────────╰X─╭●─────────────│───RY(0.00)──RZ(0.00)─── ···
5: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─────────────╰X─╭●──────────│───RY(0.00)──RZ(0.00)─── ···
6: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)────────────────╰X─╭●───────│───RY(0.00)──RZ(0.00)─── ···
7: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)───────────────────╰X─╭●────│───RY(0.00)──RZ(0.00)─── ···
8: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)──────────────────────╰X─╭●─│───RY(0.00)──RZ(0.00)─── ···
9: ─╰AngleEmbedding(M0)──RY(0.00)──R

In [8]:
# Q1 PNG
fig, ax = qml.draw_mpl(q1_circuit, decimals=2)(sample_input_q1, sample_weights_q1)
fig.set_size_inches(26, 8)
fig.suptitle("Q1 Circuit - Pure Angle Encoding (10 qubits, 2 layers)", fontsize=14, fontweight='bold')
fig.tight_layout()
q1_png = os.path.join(evidence_fig_dir, "q1_circuit_diagram.png")
fig.savefig(q1_png, dpi=200, bbox_inches="tight")
plt.close(fig)

# Q2 PNG
fig, ax = qml.draw_mpl(q2_circuit, decimals=2)(sample_input_q2, sample_weights_q2)
fig.set_size_inches(30, 10)
fig.suptitle("Q2 Circuit - Pure Angle Encoding (10 qubits, 4 layers)", fontsize=14, fontweight='bold')
fig.tight_layout()
q2_png = os.path.join(evidence_fig_dir, "q2_circuit_diagram.png")
fig.savefig(q2_png, dpi=200, bbox_inches="tight")
plt.close(fig)

# Q3 Angle Branch PNG
fig, ax = qml.draw_mpl(q3_angle_branch, decimals=2)(sample_input_q3_angle, sample_weights_q3_angle)
fig.set_size_inches(26, 8)
fig.suptitle("Q3 Angle Branch Circuit (10 qubits, 2 layers)", fontsize=14, fontweight='bold')
fig.tight_layout()
q3_angle_png = os.path.join(evidence_fig_dir, "q3_angle_branch_circuit_diagram.png")
fig.savefig(q3_angle_png, dpi=200, bbox_inches="tight")
plt.close(fig)

# Q3 Amplitude Branch PNG
fig, ax = qml.draw_mpl(q3_amplitude_branch, decimals=2)(sample_input_q3_amp, sample_weights_q3_amp)
fig.set_size_inches(16, 6)
fig.suptitle("Q3 Amplitude Branch Circuit (4 qubits, 2 layers)", fontsize=14, fontweight='bold')
fig.tight_layout()
q3_amp_png = os.path.join(evidence_fig_dir, "q3_amplitude_branch_circuit_diagram.png")
fig.savefig(q3_amp_png, dpi=200, bbox_inches="tight")
plt.close(fig)

# Deney klasorlerine kopyala
shutil.copy(q1_png, os.path.join(q12_fig_dir, "q1_circuit_diagram.png"))
shutil.copy(q2_png, os.path.join(q12_fig_dir, "q2_circuit_diagram.png"))
shutil.copy(q3_angle_png, os.path.join(q3_fig_dir, "q3_angle_branch_circuit_diagram.png"))
shutil.copy(q3_amp_png, os.path.join(q3_fig_dir, "q3_amplitude_branch_circuit_diagram.png"))

print("PNG devre cizimleri kaydedildi.")
print("-", q1_png)
print("-", q2_png)
print("-", q3_angle_png)
print("-", q3_amp_png)

/tmp/ipykernel_5098/2753397716.py:5: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_5098/2753397716.py:14: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_5098/2753397716.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_5098/2753397716.py:32: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


PNG devre cizimleri kaydedildi.
- /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures/q1_circuit_diagram.png
- /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures/q2_circuit_diagram.png
- /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures/q3_angle_branch_circuit_diagram.png
- /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures/q3_amplitude_branch_circuit_diagram.png


In [9]:
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

def add_box(x, y, w, h, text, color):
    box = patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02",
        linewidth=1.5,
        edgecolor="black",
        facecolor=color
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=11, fontweight='bold')

# Sol input
add_box(0.03, 0.42, 0.14, 0.16, "Input\n10 Features", "#d6eaf8")

# Angle branch
add_box(0.24, 0.62, 0.18, 0.15, "Angle Branch\n10 qubits\nAngleEmbedding\n2 layers", "#d5f5e3")
add_box(0.48, 0.62, 0.14, 0.15, "10 Readouts\n⟨Z⟩", "#fcf3cf")

# Amplitude preprocessing
add_box(0.22, 0.18, 0.20, 0.15, "Pad to 16\nL2 Normalize", "#f9e79f")
add_box(0.48, 0.18, 0.16, 0.15, "Amplitude Branch\n4 qubits\n2 layers", "#fadbd8")
add_box(0.68, 0.18, 0.12, 0.15, "4 Readouts\n⟨Z⟩", "#f5cba7")

# Fusion ve head
add_box(0.72, 0.52, 0.14, 0.16, "Concatenate\n10 + 4 = 14", "#e8daef")
add_box(0.88, 0.52, 0.10, 0.16, "Classical\nHead\n7 classes", "#d6dbdf")

# Oklar
ax.annotate("", xy=(0.24, 0.70), xytext=(0.17, 0.50), arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(0.22, 0.26), xytext=(0.17, 0.50), arrowprops=dict(arrowstyle="->", lw=2))

ax.annotate("", xy=(0.48, 0.70), xytext=(0.42, 0.70), arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(0.48, 0.26), xytext=(0.42, 0.26), arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(0.68, 0.26), xytext=(0.64, 0.26), arrowprops=dict(arrowstyle="->", lw=2))

ax.annotate("", xy=(0.72, 0.60), xytext=(0.62, 0.70), arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(0.72, 0.56), xytext=(0.80, 0.33), arrowprops=dict(arrowstyle="->", lw=2))
ax.annotate("", xy=(0.88, 0.60), xytext=(0.86, 0.60), arrowprops=dict(arrowstyle="->", lw=2))

plt.title("Q3 Architecture - Dual-Branch Hybrid Encoding Fusion Model",
          fontsize=14, fontweight='bold')

q3_arch_png = os.path.join(evidence_fig_dir, "q3_architecture_diagram.png")
plt.savefig(q3_arch_png, dpi=200, bbox_inches="tight")
plt.close()

shutil.copy(q3_arch_png, os.path.join(q3_fig_dir, "q3_architecture_diagram.png"))

print("Q3 architecture diagram kaydedildi:")
print("-", q3_arch_png)

Q3 architecture diagram kaydedildi:
- /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/figures/q3_architecture_diagram.png


In [10]:
evidence_text = """
QUANTUM EXECUTION EVIDENCE
==========================

This project implemented quantum machine learning models using PennyLane's default.qubit simulator.

Important clarification:
- The models were NOT executed on real quantum hardware.
- The models WERE executed as valid quantum circuits on a statevector-based quantum simulator.
- Therefore, the work constitutes a simulator-based quantum machine learning study.

Implemented quantum models:
1. Q1:
   - 10 qubits
   - Pure angle encoding
   - 2 variational layers
   - Ring CNOT entanglement
   - All-qubit readout
   - Classical classification head

2. Q2:
   - 10 qubits
   - Pure angle encoding
   - 4 variational layers
   - Ring CNOT entanglement
   - All-qubit readout
   - Classical classification head

3. Q3:
   - Dual-branch hybrid-encoding architecture
   - Angle branch: 10 qubits, angle encoding, 2 layers
   - Amplitude branch: 4 qubits, amplitude encoding, 2 layers
   - Fusion of both branches
   - Classical classification head

Why this counts as quantum:
- Quantum states were prepared using AngleEmbedding and AmplitudeEmbedding
- Quantum gates (RY, RZ, CNOT) were applied
- Quantum measurements (PauliZ expectation values) were collected
- Classical optimization updated trainable quantum parameters during training

Saved evidence:
- Python circuit definitions (.py)
- ASCII circuit renderings (.txt)
- Circuit diagrams (.png)
- Experiment metadata and summaries
- Trained model weights (.pt)
- Training histories and evaluation tables (.csv)

Conclusion:
The project constitutes a valid hybrid quantum-classical machine learning workflow executed on a quantum simulator.
""".strip()

evidence_txt_path = os.path.join(evidence_note_dir, "quantum_execution_evidence.txt")
with open(evidence_txt_path, "w", encoding="utf-8") as f:
    f.write(evidence_text)

print("Kaydedildi:", evidence_txt_path)
print()
print(evidence_text[:1500])

Kaydedildi: /content/drive/MyDrive/Obesity_Quantum_Project/quantum_circuit_evidence/notes/quantum_execution_evidence.txt

QUANTUM EXECUTION EVIDENCE

This project implemented quantum machine learning models using PennyLane's default.qubit simulator.

Important clarification:
- The models were NOT executed on real quantum hardware.
- The models WERE executed as valid quantum circuits on a statevector-based quantum simulator.
- Therefore, the work constitutes a simulator-based quantum machine learning study.

Implemented quantum models:
1. Q1:
   - 10 qubits
   - Pure angle encoding
   - 2 variational layers
   - Ring CNOT entanglement
   - All-qubit readout
   - Classical classification head

2. Q2:
   - 10 qubits
   - Pure angle encoding
   - 4 variational layers
   - Ring CNOT entanglement
   - All-qubit readout
   - Classical classification head

3. Q3:
   - Dual-branch hybrid-encoding architecture
   - Angle branch: 10 qubits, angle encoding, 2 layers
   - Amplitude branch: 4 qubits

In [11]:
print("=== EVIDENCE / FIGURES ===")
for f in sorted(os.listdir(evidence_fig_dir)):
    print("-", f)

print("\n=== EVIDENCE / CODE ===")
for f in sorted(os.listdir(evidence_code_dir)):
    print("-", f)

print("\n=== EVIDENCE / NOTES ===")
for f in sorted(os.listdir(evidence_note_dir)):
    print("-", f)

=== EVIDENCE / FIGURES ===
- q1_circuit_diagram.png
- q2_circuit_diagram.png
- q3_amplitude_branch_circuit_diagram.png
- q3_angle_branch_circuit_diagram.png
- q3_architecture_diagram.png

=== EVIDENCE / CODE ===
- q1_circuit_definition.py
- q2_circuit_definition.py
- q3_amplitude_branch_definition.py
- q3_angle_branch_definition.py

=== EVIDENCE / NOTES ===
- q1_circuit_ascii.txt
- q2_circuit_ascii.txt
- q3_amplitude_branch_ascii.txt
- q3_angle_branch_ascii.txt
- quantum_execution_evidence.txt
